#SETUP

In [6]:
import os
from dotenv import load_dotenv, find_dotenv
PERPLEXITY_KEY = os.getenv("PERPLEXITY_API_KEY")
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"
env_path = find_dotenv()
load_dotenv(env_path, override=True)


True

#LOGGING A TRACE

In [29]:
PPLX_BASE = "https://api.perplexity.ai"
MODEL = "sonar-pro"

def call_perplexity(messages, model=MODEL, temperature=0.0, timeout=60):
    payload = {"model": model, "messages": messages, "temperature": temperature}
    r = requests.post(f"{PPLX_BASE}/chat/completions",
                      headers={"Authorization": f"Bearer {PPLX_API_KEY}", "Content-Type": "application/json"},
                      json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    ch = data.get("choices", [{}])[0]
    if isinstance(ch.get("message"), dict):
        return ch["message"].get("content", "")
    return ch.get("text", "") or json.dumps(data)


In [30]:
def make_structured_prompt(template, system="You are concise and factual."):
    def invoke(inputs):
        text = template.format(**inputs)
        return {"messages":[{"role":"system","content":system},{"role":"user","content":text}]}
    return SimpleNamespace(invoke=invoke)


In [31]:
from langsmith import Client
from requests.exceptions import HTTPError

client = Client() if LANGSMITH_API_KEY else None

def get_or_create_dataset_id(name):
    if client is None: 
        raise RuntimeError("LANGSMITH_API_KEY not set")
    try:
        ds = client.create_dataset(dataset_name=name)
        return ds["id"] if isinstance(ds, dict) else ds.id
    except Exception as e:
        txt = str(e).lower()
        if "already exists" in txt or "409" in txt:
            all_ds = client.list_datasets()
            for d in all_ds:
                if isinstance(d, dict):
                    if d.get("name") == name: return d.get("id")
                else:
                    if getattr(d, "name", None) == name: return getattr(d, "id", None)
            raise RuntimeError(f"Dataset {name} exists but could not be resolved") from e
        raise


In [32]:
def upload_example(dataset_id, input_obj, output_obj):
    if client is None:
        raise RuntimeError("LANGSMITH_API_KEY not set")
    client.create_examples(inputs=[input_obj], outputs=[output_obj], dataset_id=dataset_id)
    return dataset_id


In [33]:
template = "Answer in {language} in one or two sentences.\nContext: {context}\nAttachment: {attachment}\nQuestion: {question}"
prompt = make_structured_prompt(template)

inputs = {
    "question": "Summarize the notebook for a beginner.",
    "language": "English",
    "context": "Prompt Hub -> hydrate -> call model -> save results",
    "attachment": FILE_PATH
}

hydrated = prompt.invoke(inputs)
messages = hydrated["messages"]
print("Hydrated messages:", messages)

response = call_perplexity(messages)
print("\nPerplexity response:\n", response)

if LANGSMITH_API_KEY:
    ds_id = get_or_create_dataset_id("Perplexity-RAG-Results")
    upload_example(ds_id, inputs, {"output": response})
    print("Saved to dataset:", ds_id)
else:
    print("LANGSMITH_API_KEY not set — skipped save.")


Hydrated messages: [{'role': 'system', 'content': 'You are concise and factual.'}, {'role': 'user', 'content': 'Answer in English in one or two sentences.\nContext: Prompt Hub -> hydrate -> call model -> save results\nAttachment: /mnt/data/prompt_engineering_lifecycle.ipynb\nQuestion: Summarize the notebook for a beginner.'}]

Perplexity response:
 The notebook demonstrates a simple prompt engineering workflow: it shows how to set up a prompt, send it to a language model, and save the results, guiding beginners through the basic steps of interacting with AI models using code.
Saved to dataset: ca4e41a1-be3d-498e-9860-5480bb55ecc9
